# Comparing Fusion Rate in MDAMB231s and HCC1806s Using FACS after 4-days of co-culture

- Use environment.yml

## Import Necessary Modules

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.stats import ttest_ind
from scipy.stats import ttest_rel
from scipy.stats import tukey_hsd
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import matplotlib.cm as cm
from matplotlib import font_manager
from matplotlib import rcParams
import pathlib

## Functions

In [ ]:
# for font_path in pathlib.Path('/stor/work/Brock/kennedy/fonts/arial').glob('*.TTF'):
#     font_manager.fontManager.addfont(str(font_path))

# arial_n = {'fontproperties':font_manager.FontProperties(fname="../misc/arial/ARIALN.TTF")}
# arial_nb = {'fontproperties':font_manager.FontProperties(fname="../misc/arial/ARIALNB.TTF")}

# assert 'Arial' in [f.name for f in font_manager.fontManager.ttflist]

# rcParams['font.family'] = 'Arial'

In [ ]:
def stats_and_plot(dataframe, alpha=0.05, xlabel=None, ylabel=None, title=None,
                    sig_bars=True, point_label=False, fig_size=(6, 6), font_sz=18,
                    save_svg=False, force_ttest=False, ylim_pad=0.05,
                    broken_axis=False, break_pad_frac=0.25, break_height_ratio=(1, 1)):
    """
    Perform t-tests between data in each column of the DataFrame
    and create a box and whisker plot with indicators for statistical significance.

    Parameters:
    dataframe (pd.DataFrame): The input DataFrame.
    alpha (float, optional): The significance level for the t-tests. Default is 0.05.
    save_svg (bool): If True, saves the plot as an SVG file using the title as the filename.
    force_ttest (bool, optional): If True, skip the normality/variance checks and always
        use the "classic" parametric test, Student's unpaired t-test for 2 groups, or
        one-way ANOVA + Tukey HSD for 3+ groups, regardless of whether assumptions pass.
    ylim_pad (float, optional): Fraction of the overall data range (max - min) used as
        padding below the lowest point / above the highest sig bar. Only used when
        broken_axis is False.
    broken_axis (bool, optional): If True AND the dataframe has exactly 2 columns,
        split the y-axis into two panels (one zoomed to each group's own range) with
        a diagonal break mark, instead of one axis stretched to cover both groups.
        Useful when one group's values are much larger/smaller than the other's,
        which otherwise squashes each box into a thin sliver. Has no effect (silently
        falls back to a normal single axis, with a printed note) when there are more
        than 2 columns, since the break point is ambiguous with 3+ groups.
    break_pad_frac (float, optional): Padding around each group's own min/max as a
        fraction of that group's own range, used only when broken_axis is True.
    break_height_ratio (tuple, optional): Relative height of (top panel, bottom panel)
        when broken_axis is True. E.g. (1, 2) gives the bottom/low group twice the
        vertical room of the top/high group.

    Returns:
    pd.DataFrame: A DataFrame containing the p-values between each pair of columns.
    """
    import matplotlib as mpl
    from scipy.stats import shapiro, levene, mannwhitneyu, ttest_ind, kruskal
    from scipy.stats import f_oneway, tukey_hsd
    # Only needed for the 3+ group branch, imported lazily below so 2-group
    # runs don't require these to be installed.

    # --- Publication style settings ---
    mpl.rcParams.update({
        'font.family': 'Arial',
        'font.weight': 'bold',
        'axes.labelweight': 'bold',
        'axes.titleweight': 'bold',
        'font.size': font_sz,
        'axes.titlesize': font_sz,
        'axes.labelsize': font_sz,
        'xtick.labelsize': font_sz - 2,
        'ytick.labelsize': font_sz - 2,
        'legend.fontsize': font_sz - 4,
        'axes.linewidth': 1.5,
        'xtick.major.width': 1.5,
        'ytick.major.width': 1.5,
        'xtick.major.size': 5,
        'ytick.major.size': 5,
        'lines.linewidth': 1.5,
    })

    # Convert dataframe to numeric
    dataframe = dataframe.apply(pd.to_numeric, errors='coerce')
    num_samples = dataframe.shape[1]
    p_values_df = pd.DataFrame(columns=dataframe.columns, index=dataframe.columns)

    # broken_axis only makes sense for exactly 2 groups (unambiguous single break point)
    use_broken_axis = broken_axis and num_samples == 2
    if broken_axis and num_samples != 2:
        print("broken_axis=True is only supported for exactly 2 columns; "
              "falling back to a normal single axis.")

    # --- Assumption testing (shared by both plot modes) ---
    alpha_assumption = 0.05
    groups = [dataframe[col].dropna().values for col in dataframe.columns]

    normality_ok = all(shapiro(g).pvalue > alpha_assumption for g in groups if len(g) >= 3)
    levene_p = levene(*groups).pvalue
    variance_ok = levene_p > alpha_assumption

    print(f"Normality (Shapiro-Wilk):     {'PASS' if normality_ok else 'FAIL'}")
    print(f"Equal variance (Levene's):    {'PASS' if variance_ok else 'FAIL'}")
    if force_ttest:
        print("force_ttest=True -> assumption checks above are informational only; "
              "the classic parametric test is being used regardless.")

    # --- Statistics ---
    if num_samples == 2:
        col1, col2 = dataframe.columns[0], dataframe.columns[1]
        g1, g2 = dataframe[col1].dropna().values, dataframe[col2].dropna().values

        if force_ttest:
            test_used = "Independent samples t-test (forced)"
            _, p_value = ttest_ind(g1, g2, equal_var=True)
        elif normality_ok and variance_ok:
            test_used = "Independent samples t-test"
            _, p_value = ttest_ind(g1, g2, equal_var=True)
        elif normality_ok and not variance_ok:
            test_used = "Welch's t-test"
            _, p_value = ttest_ind(g1, g2, equal_var=False)
        else:
            test_used = "Mann-Whitney U test"
            _, p_value = mannwhitneyu(g1, g2, alternative='two-sided')

        p_values_df.loc[col1, col2] = p_value
        print(f"Test selected:                {test_used}")
        print(f"p-value:                      {p_value:.4f}")

    else:
        if force_ttest or (normality_ok and variance_ok):
            test_used = "One-way ANOVA + Tukey HSD" + (" (forced)" if force_ttest else "")
            _, p_omnibus = f_oneway(*groups)
            res = tukey_hsd(*groups)
            p_values_df = pd.DataFrame(res.pvalue, columns=dataframe.columns, index=dataframe.columns)

        elif normality_ok and not variance_ok:
            import pingouin as pg
            test_used = "Welch's ANOVA + Games-Howell"
            melted = dataframe.melt(var_name='group', value_name='value').dropna()
            welch_result = pg.welch_anova(data=melted, dv='value', between='group')
            p_omnibus = welch_result['p_unc'].values[0]
            gh = pg.pairwise_gameshowell(data=melted, dv='value', between='group')
            p_values_df = pd.DataFrame(np.nan, columns=dataframe.columns, index=dataframe.columns)
            for _, row in gh.iterrows():
                p_values_df.loc[row['A'], row['B']] = row['pval']
                p_values_df.loc[row['B'], row['A']] = row['pval']

        else:
            import scikit_posthocs as sp
            test_used = "Kruskal-Wallis + Dunn's post-hoc (Bonferroni)"
            _, p_omnibus = kruskal(*groups)
            melted = dataframe.melt(var_name='group', value_name='value').dropna()
            dunn = sp.posthoc_dunn(melted, val_col='value', group_col='group', p_adjust='bonferroni')
            p_values_df = dunn

        print(f"Test selected:                {test_used}")
        print(f"Omnibus p-value:              {p_omnibus:.2e}")

    plot_title = title if title else 'Boxplot with Statistical Significance Indicators'

    # =========================================================================
    # PLOT MODE 1: broken axis (2 groups only)
    # =========================================================================
    if use_broken_axis:
        col1, col2 = dataframe.columns[0], dataframe.columns[1]
        g1, g2 = dataframe[col1].dropna().values, dataframe[col2].dropna().values

        low_col, low_vals, high_col, high_vals = (
            (col1, g1, col2, g2) if np.mean(g1) < np.mean(g2) else (col2, g2, col1, g1)
        )

        def local_range(vals):
            lo, hi = vals.min(), vals.max()
            span = (hi - lo) if hi > lo else (abs(hi) or 1.0)
            return lo - span * break_pad_frac, hi + span * break_pad_frac

        low_bottom, low_top = local_range(low_vals)
        high_bottom, high_top = local_range(high_vals)

        fig, (ax_top, ax_bot) = plt.subplots(
            2, 1, sharex=True, figsize=fig_size,
            gridspec_kw={'height_ratios': break_height_ratio, 'hspace': 0.08}
        )

        for ax in (ax_top, ax_bot):
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            ax.spines['left'].set_linewidth(1.5)
            ax.spines['bottom'].set_linewidth(1.5)

            bp = ax.boxplot(
                [g1, g2], **{'labels': [col1, col2]},
                showmeans=True, showfliers=False, patch_artist=True,
                meanprops=dict(marker='D', markerfacecolor='black', markeredgecolor='black', markersize=5),
                medianprops=dict(color='black', linewidth=2),
                boxprops=dict(facecolor='white', color='black', linewidth=1.5),
                whiskerprops=dict(color='black', linewidth=1.5, linestyle='--'),
                capprops=dict(color='black', linewidth=1.5),
            )

            for i, (col, vals) in enumerate(zip([col1, col2], [g1, g2]), start=1):
                x = np.random.normal(i, 0.04, len(vals))
                ax.scatter(x, vals, color='dimgray', alpha=0.6, s=30, zorder=3, edgecolors='none')

                if point_label:
                    valid_indices = dataframe[col].dropna().index
                    for x_val, y_val, label in zip(x, vals, valid_indices):
                        ax.text(x_val, y_val, str(label), fontsize=8, ha='right', va='bottom')

        ax_top.set_ylim(high_bottom, high_top)
        ax_bot.set_ylim(low_bottom, low_top)

        # Hide the shared spine between panels, draw diagonal break marks
        ax_top.spines['bottom'].set_visible(False)
        ax_bot.spines['top'].set_visible(False)
        ax_top.tick_params(labeltop=False, bottom=False)
        ax_bot.xaxis.tick_bottom()

        d = 0.012
        kwargs = dict(transform=ax_top.transAxes, color='k', clip_on=False, lw=1.5)
        ax_top.plot((-d, +d), (-d, +d), **kwargs)
        ax_top.plot((1 - d, 1 + d), (-d, +d), **kwargs)
        kwargs.update(transform=ax_bot.transAxes)
        ax_bot.plot((-d, +d), (1 - d, 1 + d), **kwargs)
        ax_bot.plot((1 - d, 1 + d), (1 - d, 1 + d), **kwargs)

        # Significance bar on the top (high-value) panel
        if sig_bars:
            p_value = p_values_df.loc[col1, col2]
            if not pd.isna(p_value) and p_value < alpha:
                sig_symbol = '***' if p_value < 0.001 else ('**' if p_value < 0.01 else '*')
                y = high_top - (high_top - high_bottom) * 0.08
                ax_top.plot([1, 2], [y, y], lw=1.5, color='black')
                ax_top.text(1.5, y, sig_symbol, ha='center', va='bottom', color='black', fontsize=13)

        fig.supylabel(ylabel if ylabel else 'Growth Rate', fontsize=font_sz, fontweight='bold')
        ax_bot.set_xlabel(xlabel if xlabel else 'Samples', labelpad=10)
        fig.suptitle(plot_title, fontweight='bold')

        fig.tight_layout(rect=[0.02, 0, 1, 0.97])
        fig.subplots_adjust(left=0.18)

        if save_svg:
            fig.savefig(f"{plot_title}.svg", format='svg', bbox_inches='tight', dpi=300)

        plt.show()
        return p_values_df

    # =========================================================================
    # PLOT MODE 2: normal single axis
    # =========================================================================
    fig, ax = plt.subplots(figsize=fig_size)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(1.5)
    ax.spines['bottom'].set_linewidth(1.5)
    ax.yaxis.set_ticks_position('left')
    ax.xaxis.set_ticks_position('bottom')

    bp = ax.boxplot(
        [dataframe[col].dropna() for col in dataframe.columns],
        **{'labels': dataframe.columns},
        showmeans=True,
        showfliers=False,
        patch_artist=True,
        meanprops=dict(marker='D', markerfacecolor='black', markeredgecolor='black', markersize=5),
        medianprops=dict(color='black', linewidth=2),
        boxprops=dict(facecolor='white', color='black', linewidth=1.5),
        whiskerprops=dict(color='black', linewidth=1.5, linestyle='--'),
        capprops=dict(color='black', linewidth=1.5),
    )

    for i, col in enumerate(dataframe.columns, start=1):
        y = dataframe[col].dropna()
        x = np.random.normal(i, 0.04, len(y))
        ax.scatter(x, y, color='dimgray', alpha=0.6, s=30, zorder=3, edgecolors='none')

        if point_label:
            valid_indices = dataframe[col].dropna().index
            for x_val, y_val, label in zip(x, y, valid_indices):
                ax.text(x_val, y_val, str(label), fontsize=8, ha='right', va='bottom')

    data_min = dataframe.min().min()
    data_max = dataframe.max().max()
    data_range = data_max - data_min if data_max > data_min else abs(data_max) or 1.0

    highest_y = data_max

    if sig_bars:
        y_span_for_bars = data_range
        line_offset = y_span_for_bars * 0.09
        text_offset = y_span_for_bars * 0.02
        drawn_pairs = set()
        highest_y = data_max

        for col1 in p_values_df.columns:
            for col2 in p_values_df.index:
                p_value = p_values_df.loc[col2, col1]
                if not pd.isna(p_value) and p_value < alpha:
                    if (col1, col2) in drawn_pairs or (col2, col1) in drawn_pairs:
                        continue

                    if p_value < 0.001:
                        sig_symbol = '***'
                    elif p_value < 0.01:
                        sig_symbol = '**'
                    elif p_value < 0.05:
                        sig_symbol = '*'
                    else:
                        continue

                    x1 = dataframe.columns.get_loc(col1) + 1
                    x2 = dataframe.columns.get_loc(col2) + 1
                    y = data_max + line_offset

                    ax.plot([x1, x2], [y, y], lw=1.5, color='black')
                    ax.text((x1 + x2) * 0.5, y + text_offset, sig_symbol,
                            ha='center', va='bottom', color='black', fontsize=13)

                    drawn_pairs.add((col1, col2))
                    drawn_pairs.add((col2, col1))
                    line_offset += y_span_for_bars * 0.09
                    highest_y = y + text_offset

    bottom = data_min - data_range * ylim_pad
    top = highest_y + data_range * ylim_pad
    ax.set_ylim(bottom=bottom, top=top)

    ax.set_xlabel(xlabel if xlabel else 'Samples', labelpad=10)
    ax.set_ylabel(ylabel if ylabel else 'Growth Rate', labelpad=10)
    ax.set_title(plot_title, pad=12, fontweight='bold')

    fig.tight_layout()

    if save_svg:
        fig.savefig(f"{plot_title}.svg", format='svg', bbox_inches='tight', dpi=300)

    plt.show()
    return p_values_df

## Main Function

#### HCC1806 Fusion Population percentage

In [ ]:
HCC1806_percents_df = pd.read_csv('/stor/work/Brock/kennedy/SC_repo/data/FACSFusionRateAnalysis/HCC1806_fusionrate_percents.csv')
print(HCC1806_percents_df[:1].mean())

In [ ]:
stats_and_plot(HCC1806_percents_df, alpha=0.05, ylim_pad=0.05, sig_bars=True, save_svg=True, broken_axis=True, fig_size=(6, 6), ylabel='Fusion Cells (%)', title='Percent of double positive events in control versus co-cultured HCC1806s',force_ttest=True)

#### MDAMB231 Fusion Population percentage

In [ ]:
MDAMB231_percents_df = pd.read_csv('/stor/work/Brock/kennedy/SC_repo/data/FACSFusionRateAnalysis/MDAMB231_fusionrate_percents.csv')
print(MDAMB231_percents_df[:1].mean())

In [ ]:
stats_and_plot(MDAMB231_percents_df, alpha=0.05, sig_bars=True, save_svg=True, fig_size=(6, 6), broken_axis=True, ylabel='Fusion Cells (%)', title='Percent of double positive events in control versus co-cultured MDAMB231s',force_ttest=True)